# Searching and Sorting Comparison Lab

This lab compares algorithms using correctness checks and operation counts. The goal is to connect the code to growth-rate ideas rather than to run a timing race.


```{index} algorithm comparison lab
```

## Compare Search Counts

Count comparisons for linear search and binary search on the same sorted array.


In [ ]:
using System;

int[] sorted = { 2, 4, 6, 8, 10, 12, 14, 16 };
int target = 14;

int linearComparisons = 0;
int binaryComparisons = 0;

int linearIndex = LinearSearch(sorted, target, ref linearComparisons);
int binaryIndex = BinarySearch(sorted, target, ref binaryComparisons);

Console.WriteLine($"linear: index={linearIndex}, comparisons={linearComparisons}");
Console.WriteLine($"binary: index={binaryIndex}, comparisons={binaryComparisons}");

int LinearSearch(int[] values, int target, ref int comparisons)
{
    for (int i = 0; i < values.Length; i++)
    {
        comparisons++;
        if (values[i] == target)
        {
            return i;
        }
    }
    return -1;
}

int BinarySearch(int[] values, int target, ref int comparisons)
{
    int low = 0;
    int high = values.Length - 1;
    while (low <= high)
    {
        int middle = low + (high - low) / 2;
        comparisons++;
        if (values[middle] == target)
        {
            return middle;
        }
        if (target < values[middle])
        {
            high = middle - 1;
        }
        else
        {
            low = middle + 1;
        }
    }
    return -1;
}


```{index} sorting test
```

## Test Sorting Correctness

A sorting algorithm should handle empty arrays, one-item arrays, duplicates, already sorted arrays, and reverse-sorted arrays.


In [ ]:
using System;
using System.Linq;

int[][] tests = {
    Array.Empty<int>(),
    new[] { 4 },
    new[] { 3, 1, 2 },
    new[] { 2, 2, 1 },
    new[] { 1, 2, 3 },
    new[] { 3, 2, 1 }
};

foreach (int[] test in tests)
{
    int[] copy = test.ToArray();
    InsertionSort(copy);
    Console.WriteLine($"[{string.Join(", ", test)}] -> [{string.Join(", ", copy)}] ok={IsSorted(copy)}");
}

void InsertionSort(int[] values)
{
    for (int i = 1; i < values.Length; i++)
    {
        int current = values[i];
        int j = i - 1;
        while (j >= 0 && values[j] > current)
        {
            values[j + 1] = values[j];
            j--;
        }
        values[j + 1] = current;
    }
}

bool IsSorted(int[] values)
{
    return values.Zip(values.Skip(1), (a, b) => a <= b).All(result => result);
}


Correctness tests are not a substitute for analysis, but they catch implementation mistakes before you compare performance.


```{index} selection sort; insertion sort; bubble sort
```

## Compare Elementary Sort Counts

Count comparisons for selection sort, insertion sort, and bubble sort on the same starting data.


In [ ]:
using System;
using System.Linq;

int[] original = { 7, 3, 9, 1, 5, 2 };

Run("selection", SelectionSortCount);
Run("insertion", InsertionSortCount);
Run("bubble", BubbleSortCount);

void Run(string name, Func<int[], int> sort)
{
    int[] copy = original.ToArray();
    int comparisons = sort(copy);
    Console.WriteLine($"{name,-9}: {string.Join(", ", copy)} comparisons={comparisons}");
}

int SelectionSortCount(int[] values)
{
    int comparisons = 0;
    for (int start = 0; start < values.Length - 1; start++)
    {
        int smallest = start;
        for (int i = start + 1; i < values.Length; i++)
        {
            comparisons++;
            if (values[i] < values[smallest]) smallest = i;
        }
        Swap(values, start, smallest);
    }
    return comparisons;
}

int InsertionSortCount(int[] values)
{
    int comparisons = 0;
    for (int i = 1; i < values.Length; i++)
    {
        int current = values[i];
        int j = i - 1;
        while (j >= 0)
        {
            comparisons++;
            if (values[j] <= current) break;
            values[j + 1] = values[j];
            j--;
        }
        values[j + 1] = current;
    }
    return comparisons;
}

int BubbleSortCount(int[] values)
{
    int comparisons = 0;
    bool swapped;
    do
    {
        swapped = false;
        for (int i = 0; i < values.Length - 1; i++)
        {
            comparisons++;
            if (values[i] > values[i + 1])
            {
                Swap(values, i, i + 1);
                swapped = true;
            }
        }
    } while (swapped);
    return comparisons;
}

void Swap(int[] values, int first, int second)
{
    int temp = values[first];
    values[first] = values[second];
    values[second] = temp;
}


The counts may differ by input order. The shared lesson is that these elementary sorts still grow quadratically in the worst case.


```{index} merge sort; comparison
```

## Compare With Merge Sort Growth

Merge sort from Chapter 20 has `O(n log n)` growth, while the elementary sorts in this chapter have `O(n^2)` worst-case growth.


In [ ]:
using System;

int[] sizes = { 10, 100, 1000 };

foreach (int n in sizes)
{
    double nLogN = n * Math.Log2(n);
    int nSquared = n * n;
    Console.WriteLine($"n={n,4}  n log n={nLogN,8:F0}  n^2={nSquared,9}");
}


For small arrays, readability and constants may matter more than the asymptotic difference. For large arrays, the growth-rate difference becomes hard to ignore.


```{index} algorithm selection
```

## Write the Recommendation

A good comparison ends with a recommendation tied to the data conditions.


In [ ]:
using System;

bool dataAlreadySorted = true;
int searchCount = 500;
int itemCount = 10_000;

if (dataAlreadySorted && searchCount > 1)
{
    Console.WriteLine("Use binary search for repeated lookups in sorted data.");
}
else if (itemCount < 20)
{
    Console.WriteLine("A simple linear scan is probably clear enough.");
}
else
{
    Console.WriteLine("Consider sorting once or using a dictionary, depending on the lookup goal.");
}


Algorithm choice is contextual. State the assumptions before you state the answer.


```{rubric} Footnotes
```
[^1]: Timing benchmarks are useful, but they require careful setup: warmup, repeated runs, realistic data, and avoiding console output inside the measured region.
[^2]: This lab uses comparison counts because they reveal the algorithmic pattern without depending on a particular computer.
